# LaBSE Phase 1–2–3 Full Evaluation

Evaluate all trained models on **IN22-Conv** across all **462 directed Indic→Indic pairs**.

Models evaluated:
1. `labse_baseline` — original `sentence-transformers/LaBSE`
2. `phase1_balanced_all462` — balanced all-pair fine-tuning
3. `phase2_weak_pair_weighted` — weak-pair weighted fine-tuning
4. `phase3_weighted_distillation` — weak-pair weighted + preservation/distillation fine-tuning

The final cell creates a small results zip: `exports/phase123_eval_results_upload_this.zip`. Upload that zip back to ChatGPT.

In [ ]:
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    _guard_dir = _candidate / "scripts"
    if (_guard_dir / "import_guard.py").exists():
        if str(_guard_dir) not in sys.path:
            sys.path.insert(0, str(_guard_dir))
        break
else:
    raise RuntimeError("Could not locate scripts/import_guard.py. Run this notebook from the WSAI workspace or copy the guard module alongside it.")

from import_guard import install_pandas_guards
install_pandas_guards()


In [ ]:
# ============================================================
# Install dependencies if needed
# ============================================================

# Run only if packages are missing:
# %pip install -U sentence-transformers datasets transformers accelerate pandas numpy tqdm scikit-learn matplotlib openpyxl

In [ ]:
# ============================================================
# Imports and setup
# ============================================================

import os
import re
import json
import math
import random
import hashlib
import zipfile
from pathlib import Path
from typing import Dict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

import matplotlib.pyplot as plt

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))

In [ ]:
# ============================================================
# Project paths
# ============================================================

USE_GOOGLE_DRIVE = False  # Keep False for SSH / VS Code setup

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_DIR = Path("/content/drive/MyDrive/labse_all_pairs_indic_finetuning").resolve()
else:
    PROJECT_DIR = Path.cwd().resolve()

OUTPUT_DIR = PROJECT_DIR / "outputs"
METRICS_DIR = PROJECT_DIR / "metrics"
DATA_DIR = PROJECT_DIR / "data"
EXPORT_DIR = PROJECT_DIR / "exports"
EVAL_DIR = METRICS_DIR / "phase123_full_evaluation"
CACHE_DIR = PROJECT_DIR / "eval_cache" / "phase123_embeddings"

for d in [OUTPUT_DIR, METRICS_DIR, DATA_DIR, EXPORT_DIR, EVAL_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PHASE1_BEST_MODEL_DIR = OUTPUT_DIR / "labse_all_462_directed_pairs_balanced" / "best_model"
PHASE2_BEST_MODEL_DIR = OUTPUT_DIR / "labse_phase2_weak_pair_weighted_from_phase1" / "best_model"
PHASE3_BEST_MODEL_DIR = OUTPUT_DIR / "labse_phase3_weighted_distillation_from_phase2" / "best_model"

print("PROJECT_DIR:", PROJECT_DIR)
print("EVAL_DIR:", EVAL_DIR)
print("CACHE_DIR:", CACHE_DIR)
print("Phase 1:", PHASE1_BEST_MODEL_DIR)
print("Phase 2:", PHASE2_BEST_MODEL_DIR)
print("Phase 3:", PHASE3_BEST_MODEL_DIR)

# Auto-discover Phase 3 if run name differs.
if not (PHASE3_BEST_MODEL_DIR / "modules.json").exists():
    candidates = []
    for p in OUTPUT_DIR.rglob("best_model"):
        if (p / "modules.json").exists() and "phase3" in str(p).lower():
            candidates.append(p)
    print("Phase3 candidates:")
    for i, p in enumerate(candidates):
        print(i, p)
    if candidates:
        PHASE3_BEST_MODEL_DIR = candidates[0]
        print("Using:", PHASE3_BEST_MODEL_DIR)

MODEL_SPECS = {
    "labse_baseline": "sentence-transformers/LaBSE",
    "phase1_balanced_all462": str(PHASE1_BEST_MODEL_DIR),
    "phase2_weak_pair_weighted": str(PHASE2_BEST_MODEL_DIR),
    "phase3_weighted_distillation": str(PHASE3_BEST_MODEL_DIR),
}

valid_model_specs = {}
for name, path_or_name in MODEL_SPECS.items():
    if name == "labse_baseline":
        valid_model_specs[name] = path_or_name
        continue
    p = Path(path_or_name)
    if (p / "modules.json").exists():
        valid_model_specs[name] = path_or_name
    else:
        print(f"WARNING: Skipping {name}. Missing valid model folder: {p}")
MODEL_SPECS = valid_model_specs

print("
Models to evaluate:")
for k, v in MODEL_SPECS.items():
    print(k, "->", v)

In [ ]:
# ============================================================
# Evaluation settings
# ============================================================

MAX_SEQ_LENGTH = 128
EVAL_ENCODE_BATCH_SIZE = 512  # reduce to 256 if encoding OOMs
RETRIEVAL_CHUNK_SIZE = 512    # reduce to 256 if retrieval OOMs
COMPUTE_RETRIEVAL = True      # Accuracy@1 / Recall@5 / Recall@10 / MRR
USE_EMBEDDING_CACHE = True
FORCE_RECOMPUTE_EMBEDDINGS = False
THRESHOLD_GRID_POINTS = 200

print("MAX_SEQ_LENGTH:", MAX_SEQ_LENGTH)
print("EVAL_ENCODE_BATCH_SIZE:", EVAL_ENCODE_BATCH_SIZE)
print("RETRIEVAL_CHUNK_SIZE:", RETRIEVAL_CHUNK_SIZE)
print("COMPUTE_RETRIEVAL:", COMPUTE_RETRIEVAL)

In [ ]:
# ============================================================
# 22 Indic languages, no English
# ============================================================

LANG_CODE_MAP = {
    "asm": "asm_Beng", "ben": "ben_Beng", "brx": "brx_Deva", "doi": "doi_Deva",
    "guj": "guj_Gujr", "hin": "hin_Deva", "kan": "kan_Knda", "kas": "kas_Arab",
    "gom": "gom_Deva", "mai": "mai_Deva", "mal": "mal_Mlym", "mni": "mni_Mtei",
    "mar": "mar_Deva", "npi": "npi_Deva", "ory": "ory_Orya", "pan": "pan_Guru",
    "san": "san_Deva", "sat": "sat_Olck", "snd": "snd_Deva", "tam": "tam_Taml",
    "tel": "tel_Telu", "urd": "urd_Arab",
}
INDIC_LANGS = list(LANG_CODE_MAP.keys())
DIRECTED_PAIRS = [(src, tgt) for src in INDIC_LANGS for tgt in INDIC_LANGS if src != tgt]
print("Indic languages:", len(INDIC_LANGS))
print("Directed pairs:", len(DIRECTED_PAIRS))
assert len(DIRECTED_PAIRS) == 462
assert "eng" not in INDIC_LANGS

In [ ]:
# ============================================================
# Load IN22-Conv
# ============================================================

def load_in22_conv():
    try:
        return load_dataset("ai4bharat/IN22-Conv", "default", split="test")
    except Exception as e1:
        print("Default config load failed:", repr(e1))
        return load_dataset("ai4bharat/IN22-Conv", split="test")

conv_ds = load_in22_conv()
conv_df = conv_ds.to_pandas()
print("IN22-Conv shape:", conv_df.shape)
print("Columns:", list(conv_df.columns))

def resolve_sentence_col(short_lang: str, df: pd.DataFrame) -> str:
    code = LANG_CODE_MAP[short_lang]
    candidates = [code, f"sentence_{code}", short_lang, f"sentence_{short_lang}"]
    if short_lang == "snd":
        candidates += ["snd_Arab", "sentence_snd_Arab", "snd_Deva", "sentence_snd_Deva"]
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"No column found for {short_lang}. Tried: {candidates}")

SENTENCE_COLS = {lang: resolve_sentence_col(lang, conv_df) for lang in INDIC_LANGS}
print("
Resolved columns:")
for lang, col in SENTENCE_COLS.items():
    print(f"{lang:>3} -> {col}")

for lang, col in SENTENCE_COLS.items():
    conv_df[col] = conv_df[col].astype(str).fillna("").str.strip()
print("Rows:", len(conv_df))

In [ ]:
# ============================================================
# Embedding cache helpers
# ============================================================

def sanitize_name(x: str) -> str:
    x = str(x).replace("/", "_").replace("\\", "_").replace(":", "_")
    x = re.sub(r"[^A-Za-z0-9_.-]+", "_", x)
    return x[:120]

def stable_hash(text: str, n_chars: int = 12) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:n_chars]

def model_cache_key(model_name: str, model_path_or_name: str) -> str:
    path = Path(model_path_or_name)
    if path.exists():
        modules = path / "modules.json"
        stamp = str(modules.stat().st_mtime_ns) if modules.exists() else "nostamp"
        raw = f"{model_name}|{path.resolve()}|{stamp}|maxlen{MAX_SEQ_LENGTH}"
    else:
        raw = f"{model_name}|{model_path_or_name}|maxlen{MAX_SEQ_LENGTH}"
    return sanitize_name(model_name) + "_" + stable_hash(raw)

def embedding_cache_path(model_name: str, model_path_or_name: str, lang: str, n_rows: int):
    key = model_cache_key(model_name, model_path_or_name)
    return CACHE_DIR / f"{key}_{lang}_n{n_rows}_l{MAX_SEQ_LENGTH}.npy"

def load_model(model_path_or_name: str) -> SentenceTransformer:
    print("Loading model:", model_path_or_name)
    model = SentenceTransformer(model_path_or_name, device=DEVICE)
    model.max_seq_length = MAX_SEQ_LENGTH
    return model

def encode_all_languages(model_name: str, model_path_or_name: str, df: pd.DataFrame) -> Dict[str, np.ndarray]:
    embeddings = {}
    model = load_model(model_path_or_name)
    for lang in INDIC_LANGS:
        col = SENTENCE_COLS[lang]
        sentences = df[col].astype(str).tolist()
        cache_path = embedding_cache_path(model_name, model_path_or_name, lang, len(sentences))
        if USE_EMBEDDING_CACHE and cache_path.exists() and not FORCE_RECOMPUTE_EMBEDDINGS:
            emb = np.load(cache_path)
            print(f"Loaded cache | model={model_name} | lang={lang} | shape={emb.shape}")
        else:
            print(f"Encoding | model={model_name} | lang={lang} | n={len(sentences)}")
            emb = model.encode(
                sentences,
                batch_size=EVAL_ENCODE_BATCH_SIZE,
                show_progress_bar=True,
                convert_to_numpy=True,
                normalize_embeddings=True,
            ).astype("float32")
            if USE_EMBEDDING_CACHE:
                np.save(cache_path, emb)
                print("Saved cache:", cache_path)
        embeddings[lang] = emb
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return embeddings

In [ ]:
# ============================================================
# Metric helpers
# ============================================================

def stable_offset(key: str, n: int) -> int:
    if n <= 1:
        return 0
    h = int(hashlib.md5(key.encode("utf-8")).hexdigest(), 16)
    return (h % (n - 1)) + 1

def separation_metrics(gold: np.ndarray, random_neg: np.ndarray) -> Dict[str, float]:
    gold = np.asarray(gold, dtype=np.float32)
    random_neg = np.asarray(random_neg, dtype=np.float32)
    mean_gold = float(np.mean(gold))
    mean_random = float(np.mean(random_neg))
    gap_values = gold - random_neg
    gap = float(np.mean(gap_values))
    midpoint = (mean_gold + mean_random) / 2.0
    sens_mid = float(np.mean(gold >= midpoint))
    spec_mid = float(np.mean(random_neg < midpoint))
    bal_mid = (sens_mid + spec_mid) / 2.0

    lo = float(min(np.min(gold), np.min(random_neg)))
    hi = float(max(np.max(gold), np.max(random_neg)))
    thresholds = np.linspace(lo, hi, THRESHOLD_GRID_POINTS)
    best_f1, best_t, best_sens, best_spec = -1.0, midpoint, sens_mid, spec_mid
    for t in thresholds:
        tp = np.sum(gold >= t); fn = np.sum(gold < t)
        fp = np.sum(random_neg >= t); tn = np.sum(random_neg < t)
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-12)
        if f1 > best_f1:
            best_f1 = float(f1); best_t = float(t); best_sens = float(recall); best_spec = float(tn / max(tn + fp, 1))
    return {
        "n": int(len(gold)),
        "mean_gold_cosine": mean_gold,
        "std_gold_cosine": float(np.std(gold)),
        "mean_random_cosine": mean_random,
        "std_random_cosine": float(np.std(random_neg)),
        "cosine_gap": gap,
        "std_pairwise_gap": float(np.std(gap_values)),
        "midpoint_threshold": float(midpoint),
        "sensitivity_midpoint": sens_mid,
        "specificity_midpoint": spec_mid,
        "balanced_accuracy_midpoint": float(bal_mid),
        "best_f1": best_f1,
        "best_f1_threshold": best_t,
        "sensitivity_best_f1": best_sens,
        "specificity_best_f1": best_spec,
    }

@torch.no_grad()
def retrieval_metrics(src_emb: np.ndarray, tgt_emb: np.ndarray, chunk_size: int = 512) -> Dict[str, float]:
    n = src_emb.shape[0]
    if n == 0:
        return {"accuracy_at_1": np.nan, "recall_at_5": np.nan, "recall_at_10": np.nan, "mrr": np.nan, "mean_rank": np.nan, "median_rank": np.nan}
    src_t = torch.from_numpy(src_emb).to(DEVICE, dtype=torch.float32)
    tgt_t = torch.from_numpy(tgt_emb).to(DEVICE, dtype=torch.float32).T
    ranks_all = []
    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        scores = src_t[start:end] @ tgt_t
        row_ids = torch.arange(end - start, device=DEVICE)
        correct_cols = torch.arange(start, end, device=DEVICE)
        correct_scores = scores[row_ids, correct_cols]
        ranks = 1 + torch.sum(scores > correct_scores[:, None], dim=1)
        ranks_all.append(ranks.detach().cpu().numpy().astype(np.int32))
    ranks = np.concatenate(ranks_all)
    return {
        "accuracy_at_1": float(np.mean(ranks <= 1)),
        "recall_at_5": float(np.mean(ranks <= 5)),
        "recall_at_10": float(np.mean(ranks <= 10)),
        "mrr": float(np.mean(1.0 / ranks)),
        "mean_rank": float(np.mean(ranks)),
        "median_rank": float(np.median(ranks)),
    }

def valid_pair_indices(df: pd.DataFrame, src: str, tgt: str) -> np.ndarray:
    src_col = SENTENCE_COLS[src]; tgt_col = SENTENCE_COLS[tgt]
    mask = (df[src_col].astype(str).str.len().values > 0) & (df[tgt_col].astype(str).str.len().values > 0)
    return np.where(mask)[0]

def evaluate_model_pair(model_name: str, embeddings: Dict[str, np.ndarray], src: str, tgt: str, df: pd.DataFrame) -> Dict[str, float]:
    idx = valid_pair_indices(df, src, tgt)
    src_emb = embeddings[src][idx]
    tgt_emb = embeddings[tgt][idx]
    n = len(idx)
    gold = np.sum(src_emb * tgt_emb, axis=1)
    shift = stable_offset(f"{model_name}|{src}->{tgt}|random_negative", n)
    random_neg = np.sum(src_emb * np.roll(tgt_emb, shift=shift, axis=0), axis=1)
    metrics = separation_metrics(gold, random_neg)
    metrics.update({"model": model_name, "src_lang": src, "tgt_lang": tgt, "directed_pair": f"{src}->{tgt}", "random_shift": int(shift)})
    if COMPUTE_RETRIEVAL:
        metrics.update(retrieval_metrics(src_emb, tgt_emb, chunk_size=RETRIEVAL_CHUNK_SIZE))
    return metrics

In [ ]:
# ============================================================
# Run full evaluation
# ============================================================

all_rows = []
for model_name, model_path_or_name in MODEL_SPECS.items():
    print("
" + "=" * 100)
    print("Evaluating model:", model_name)
    print("=" * 100)
    embeddings = encode_all_languages(model_name, model_path_or_name, conv_df)
    model_rows = []
    for src, tgt in tqdm(DIRECTED_PAIRS, desc=f"Pairs | {model_name}"):
        model_rows.append(evaluate_model_pair(model_name, embeddings, src, tgt, conv_df))
    model_df = pd.DataFrame(model_rows)
    model_out = EVAL_DIR / f"{model_name}_by_pair.csv"
    model_df.to_csv(model_out, index=False)
    print("Saved:", model_out)
    all_rows.extend(model_rows)
    del embeddings
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

by_pair_df = pd.DataFrame(all_rows)
by_pair_path = EVAL_DIR / "phase123_eval_by_pair.csv"
by_pair_df.to_csv(by_pair_path, index=False)
print("Saved full pair-level evaluation:", by_pair_path)
print("Shape:", by_pair_df.shape)
display(by_pair_df.head())

In [ ]:
# ============================================================
# Model-level summary
# ============================================================

metric_cols = [
    "cosine_gap", "mean_gold_cosine", "mean_random_cosine",
    "sensitivity_midpoint", "specificity_midpoint", "balanced_accuracy_midpoint", "best_f1",
]
if COMPUTE_RETRIEVAL:
    metric_cols += ["accuracy_at_1", "recall_at_5", "recall_at_10", "mrr", "mean_rank", "median_rank"]

summary_df = by_pair_df.groupby("model")[metric_cols].mean().reset_index()
std_cols = ["cosine_gap"] + (["accuracy_at_1"] if COMPUTE_RETRIEVAL else ["balanced_accuracy_midpoint"])
std_df = by_pair_df.groupby("model")[std_cols].std().reset_index()
std_df = std_df.rename(columns={"cosine_gap": "std_cosine_gap_across_pairs", "accuracy_at_1": "std_accuracy_at_1_across_pairs", "balanced_accuracy_midpoint": "std_balanced_accuracy_across_pairs"})
summary_df = summary_df.merge(std_df, on="model", how="left")
sort_col = "accuracy_at_1" if COMPUTE_RETRIEVAL else "cosine_gap"
summary_df = summary_df.sort_values(sort_col, ascending=False)
summary_path = EVAL_DIR / "phase123_eval_summary_by_model.csv"
summary_df.to_csv(summary_path, index=False)
print("Saved:", summary_path)
display(summary_df)

In [ ]:
# ============================================================
# Deltas vs baseline and Phase 1
# ============================================================

def make_delta_table(df: pd.DataFrame, reference_model: str, suffix: str):
    ref = df[df["model"] == reference_model].copy()
    if ref.empty:
        print("Reference not found:", reference_model)
        return pd.DataFrame()
    ref = ref.set_index("directed_pair")
    rows = []
    for model in sorted(df["model"].unique()):
        if model == reference_model:
            continue
        curr = df[df["model"] == model].copy().set_index("directed_pair")
        common = curr.index.intersection(ref.index)
        for pair in common:
            row = {"model": model, "reference_model": reference_model, "directed_pair": pair, "src_lang": curr.loc[pair, "src_lang"], "tgt_lang": curr.loc[pair, "tgt_lang"]}
            for m in metric_cols:
                if m in curr.columns and m in ref.columns:
                    row[f"delta_{m}"] = float(curr.loc[pair, m] - ref.loc[pair, m])
                    row[f"model_{m}"] = float(curr.loc[pair, m])
                    row[f"reference_{m}"] = float(ref.loc[pair, m])
            rows.append(row)
    delta_df = pd.DataFrame(rows)
    out_path = EVAL_DIR / f"phase123_delta_vs_{suffix}.csv"
    delta_df.to_csv(out_path, index=False)
    print("Saved:", out_path, "shape:", delta_df.shape)
    return delta_df

delta_vs_baseline = make_delta_table(by_pair_df, "labse_baseline", "labse_baseline")
delta_vs_phase1 = make_delta_table(by_pair_df, "phase1_balanced_all462", "phase1")

In [ ]:
# ============================================================
# Weak-pair bucket analysis
# ============================================================

baseline_df = by_pair_df[by_pair_df["model"] == "labse_baseline"].copy()
bucket_metric = "accuracy_at_1" if COMPUTE_RETRIEVAL and "accuracy_at_1" in baseline_df.columns else "cosine_gap"
print("Bucketing pairs by baseline metric:", bucket_metric)
q25 = baseline_df[bucket_metric].quantile(0.25)
q75 = baseline_df[bucket_metric].quantile(0.75)

def assign_bucket(x):
    if x <= q25:
        return "weak_bottom_25pct"
    elif x >= q75:
        return "strong_top_25pct"
    return "middle_50pct"

pair_bucket = baseline_df[["directed_pair", "src_lang", "tgt_lang", bucket_metric]].copy()
pair_bucket["baseline_bucket"] = pair_bucket[bucket_metric].apply(assign_bucket)
bucketed = by_pair_df.merge(pair_bucket[["directed_pair", "baseline_bucket"]], on="directed_pair", how="left")

bucket_summary = bucketed.groupby(["model", "baseline_bucket"])[metric_cols].mean().reset_index().sort_values(["baseline_bucket", "model"])
bucket_summary_path = EVAL_DIR / "phase123_weak_bucket_summary.csv"
bucket_summary.to_csv(bucket_summary_path, index=False)
weak_pairs_path = EVAL_DIR / "phase123_baseline_weak_pairs_bottom25.csv"
pair_bucket[pair_bucket["baseline_bucket"] == "weak_bottom_25pct"].sort_values(bucket_metric).to_csv(weak_pairs_path, index=False)
print("Saved:", bucket_summary_path)
print("Saved:", weak_pairs_path)
display(bucket_summary)

In [ ]:
# ============================================================
# Source-language and target-language summaries
# ============================================================

lang_metric_cols = ["cosine_gap", "sensitivity_midpoint", "specificity_midpoint", "balanced_accuracy_midpoint"]
if COMPUTE_RETRIEVAL:
    lang_metric_cols += ["accuracy_at_1", "recall_at_10", "mrr"]

src_summary = by_pair_df.groupby(["model", "src_lang"])[lang_metric_cols].mean().reset_index()
tgt_summary = by_pair_df.groupby(["model", "tgt_lang"])[lang_metric_cols].mean().reset_index()

src_path = EVAL_DIR / "phase123_source_language_summary.csv"
tgt_path = EVAL_DIR / "phase123_target_language_summary.csv"
src_summary.to_csv(src_path, index=False)
tgt_summary.to_csv(tgt_path, index=False)
print("Saved:", src_path)
print("Saved:", tgt_path)
display(src_summary.head())
display(tgt_summary.head())

In [ ]:
# ============================================================
# Decision table vs Phase 1
# ============================================================

summary_indexed = summary_df.set_index("model")
decision_rows = []
if "phase1_balanced_all462" in summary_indexed.index:
    ref = summary_indexed.loc["phase1_balanced_all462"]
    for model in summary_indexed.index:
        row = {"model": model}
        for m in metric_cols:
            if m in summary_indexed.columns:
                row[m] = float(summary_indexed.loc[model, m])
                row[f"delta_vs_phase1_{m}"] = float(summary_indexed.loc[model, m] - ref[m])
        decision_rows.append(row)

decision_df = pd.DataFrame(decision_rows)
if not decision_df.empty:
    if COMPUTE_RETRIEVAL and "delta_vs_phase1_accuracy_at_1" in decision_df.columns:
        decision_df["beats_phase1_accuracy"] = decision_df["delta_vs_phase1_accuracy_at_1"] > 0
    decision_df["beats_phase1_gap"] = decision_df["delta_vs_phase1_cosine_gap"] > 0
    decision_df["specificity_drop_vs_phase1"] = decision_df["delta_vs_phase1_specificity_midpoint"]
    decision_df["safe_specificity"] = decision_df["specificity_drop_vs_phase1"] >= -0.01

decision_path = EVAL_DIR / "phase123_decision_table_vs_phase1.csv"
decision_df.to_csv(decision_path, index=False)
print("Saved:", decision_path)
display(decision_df)

In [ ]:
# ============================================================
# Heatmaps: pair-level delta vs Phase 1
# ============================================================

def plot_delta_heatmap(delta_df: pd.DataFrame, model_name: str, delta_col: str, out_name: str):
    if delta_df is None or delta_df.empty:
        return
    sub = delta_df[delta_df["model"] == model_name].copy()
    if sub.empty or delta_col not in sub.columns:
        print("No heatmap data for:", model_name, delta_col)
        return
    pivot = sub.pivot(index="src_lang", columns="tgt_lang", values=delta_col)
    pivot = pivot.reindex(index=INDIC_LANGS, columns=INDIC_LANGS)
    plt.figure(figsize=(12, 9))
    plt.imshow(pivot.values, aspect="auto")
    plt.colorbar(label=delta_col)
    plt.xticks(range(len(INDIC_LANGS)), INDIC_LANGS, rotation=90)
    plt.yticks(range(len(INDIC_LANGS)), INDIC_LANGS)
    plt.title(f"{model_name}: {delta_col} vs Phase 1")
    plt.xlabel("Target language")
    plt.ylabel("Source language")
    plt.tight_layout()
    out_path = EVAL_DIR / out_name
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved heatmap:", out_path)

for model_name in ["phase2_weak_pair_weighted", "phase3_weighted_distillation"]:
    plot_delta_heatmap(delta_vs_phase1, model_name, "delta_cosine_gap", f"heatmap_{model_name}_delta_cosine_gap_vs_phase1.png")
    if COMPUTE_RETRIEVAL:
        plot_delta_heatmap(delta_vs_phase1, model_name, "delta_accuracy_at_1", f"heatmap_{model_name}_delta_accuracy_at_1_vs_phase1.png")

In [ ]:
# ============================================================
# Compact text summary to copy back to ChatGPT
# ============================================================

summary_lines = []
summary_lines.append("PHASE 1-2-3 EVALUATION SUMMARY")
summary_lines.append("=" * 60)
for _, row in summary_df.iterrows():
    parts = [
        f"model={row['model']}",
        f"cosine_gap={row.get('cosine_gap', float('nan')):.4f}",
        f"sensitivity={row.get('sensitivity_midpoint', float('nan')):.4f}",
        f"specificity={row.get('specificity_midpoint', float('nan')):.4f}",
        f"balanced_acc={row.get('balanced_accuracy_midpoint', float('nan')):.4f}",
    ]
    if COMPUTE_RETRIEVAL:
        parts += [
            f"accuracy@1={row.get('accuracy_at_1', float('nan')):.4f}",
            f"recall@10={row.get('recall_at_10', float('nan')):.4f}",
            f"mrr={row.get('mrr', float('nan')):.4f}",
        ]
    summary_lines.append(" | ".join(parts))
summary_lines.append("")
summary_lines.append("UPLOAD THIS ZIP TO CHATGPT: exports/phase123_eval_results_upload_this.zip")
summary_text = "
".join(summary_lines)
summary_txt_path = EVAL_DIR / "phase123_summary_for_chatgpt.txt"
summary_txt_path.write_text(summary_text, encoding="utf-8")
print(summary_text)
print("Saved:", summary_txt_path)

In [ ]:
# ============================================================
# Create small results zip for uploading back to ChatGPT
# This does NOT include model weights.
# ============================================================

zip_path = EXPORT_DIR / "phase123_eval_results_upload_this.zip"
include_suffixes = {".csv", ".json", ".txt", ".png", ".jpg", ".jpeg", ".xlsx"}
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in EVAL_DIR.rglob("*"):
        if file.is_file() and file.suffix.lower() in include_suffixes:
            z.write(file, arcname=file.relative_to(EVAL_DIR))
size_mb = zip_path.stat().st_size / (1024**2)
print("Created upload zip:", zip_path)
print("Size MB:", round(size_mb, 2))
print("
Upload this zip back to ChatGPT:")
print(zip_path)